In [1]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)
data_dir = "/content/drive/MyDrive"

Mounted at /content/drive


In [2]:
import os
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler

import matplotlib.pyplot as plt

from PIL import Image
import torchvision
from torchvision import transforms, datasets

from torch.utils.data import (
    DataLoader,
    random_split
)

In [3]:
import shutil

if not os.path.exists("/content/dataset_split"):

    shutil.copytree(
        "/content/drive/MyDrive/dataset_split",
        "/content/dataset_split"
    )

print("Dataset skopiowany lokalnie")

Dataset skopiowany lokalnie


In [4]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])



train_dir = "/content/dataset_split/train"
val_dir = "/content/dataset_split/val"
test_dir = "/content/dataset_split/test"

train_dataset = datasets.ImageFolder(
    train_dir,
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    val_dir,
    transform=test_transform
)

test_dataset = datasets.ImageFolder(
    test_dir,
    transform=test_transform
)

# dataloadery
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=2,
    persistent_workers=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=2,
    persistent_workers=True
)

# klasy
class_names = train_dataset.classes

print("Klasy:")
print(class_names)

print(f"\nTrain images: {len(train_dataset)}")
print(f"Val images: {len(val_dataset)}")
print(f"Test images: {len(test_dataset)}")

Klasy:
['APSK128', 'APSK16', 'APSK32', 'APSK64', 'ASK4', 'ASK8', 'BPSK', 'GMSK', 'PSK16', 'PSK32', 'PSK8', 'QAM128', 'QAM16', 'QAM256', 'QAM32', 'QAM64', 'QPSK']

Train images: 18370
Val images: 3923
Test images: 3923


In [9]:
device = torch.device("cuda")

from torchvision.models import vit_b_16, ViT_B_16_Weights

model = vit_b_16(weights=ViT_B_16_Weights.IMAGENET1K_V1)
model.heads = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(768, 256),
    nn.GELU(),
    nn.Linear(256, len(train_dataset.classes))
)
model = model.to(device)

In [6]:
print(model)

VisionTransformer(
  (conv_proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
  (encoder): Encoder(
    (dropout): Dropout(p=0.0, inplace=False)
    (layers): Sequential(
      (encoder_layer_0): EncoderBlock(
        (ln_1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (self_attention): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
        )
        (dropout): Dropout(p=0.0, inplace=False)
        (ln_2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (mlp): MLPBlock(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU(approximate='none')
          (2): Dropout(p=0.0, inplace=False)
          (3): Linear(in_features=3072, out_features=768, bias=True)
          (4): Dropout(p=0.0, inplace=False)
        )
      )
      (encoder_layer_1): EncoderBlock(
        (ln_1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
        (self_a

In [10]:
for param in model.parameters():
    param.requires_grad = False

for param in model.heads.parameters():
    param.requires_grad = True

batch_size = 64

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=4,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)


test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)


criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.heads.parameters(),
    lr=3e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',
    factor=0.5,
    patience=2
)

In [11]:
num_epochs = 30
best_val_accuracy = 0.0

train_losses = []
train_accuracies = []

val_losses = []
val_accuracies = []

for epoch in range(num_epochs):
    if epoch == 5:

        print("Stage 2")

        for param in model.encoder.layers.encoder_layer_11.parameters():
            param.requires_grad = True

        optimizer = optim.Adam([
            {
                'params': model.encoder.layers.encoder_layer_11.parameters(),
                'lr': 1e-5
            },
            {
                'params': model.heads.parameters(),
                'lr': 3e-4
            }
        ])

        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode='max',
            factor=0.5,
            patience=2
        )


    if epoch == 10:

        print("Stage 3")

        for param in model.encoder.layers.encoder_layer_10.parameters():
            param.requires_grad = True

        optimizer = optim.Adam([
            {
                'params': model.encoder.layers.encoder_layer_10.parameters(),
                'lr': 5e-6
            },
            {
                'params': model.encoder.layers.encoder_layer_11.parameters(),
                'lr': 1e-5
            },
            {
                'params': model.heads.parameters(),
                'lr': 1e-4
            }
        ])

        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode='max',
            factor=0.5,
            patience=2
        )

    print(f"\nEpoka {epoch+1}/{num_epochs}")



    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item() * images.size(0)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item()

    train_loss = running_loss / total
    train_accuracy = correct / total

    train_losses.append(train_loss)
    train_accuracies.append(train_accuracy)



    model.eval()

    val_running_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            val_running_loss += loss.item() * images.size(0)

            _, predicted = torch.max(outputs, 1)

            val_total += labels.size(0)

            val_correct += (predicted == labels).sum().item()

    val_loss = val_running_loss / val_total
    val_accuracy = val_correct / val_total

    val_losses.append(val_loss)
    val_accuracies.append(val_accuracy)



    print(
        f"Train loss: {train_loss:.4f} | "
        f"Train acc: {train_accuracy:.4f}"
    )

    print(
        f"Val loss:   {val_loss:.4f} | "
        f"Val acc:   {val_accuracy:.4f}"
    )



    torch.save(
        model.state_dict(),
        "/content/drive/MyDrive/runs_vit_b_16/model_current_vit_b_16.pth"
    )

    if val_accuracy > best_val_accuracy:

        best_val_accuracy = val_accuracy

        torch.save(
            model.state_dict(),
            "/content/drive/MyDrive/runs_vit_b_16/model_best_vit_b_16.pth"
        )

        print("Zapisano najlepszy model")

    scheduler.step(val_accuracy)

print(f"Best accuracy: {best_val_accuracy:.4f}")


Epoka 1/30
Train loss: 1.4284 | Train acc: 0.5359
Val loss:   1.8125 | Val acc:   0.5055
Zapisano najlepszy model

Epoka 2/30
Train loss: 0.8691 | Train acc: 0.7034
Val loss:   1.7533 | Val acc:   0.5277
Zapisano najlepszy model

Epoka 3/30
Train loss: 0.7408 | Train acc: 0.7378
Val loss:   1.6960 | Val acc:   0.5432
Zapisano najlepszy model

Epoka 4/30
Train loss: 0.6764 | Train acc: 0.7547
Val loss:   1.7046 | Val acc:   0.5509
Zapisano najlepszy model

Epoka 5/30
Train loss: 0.6196 | Train acc: 0.7752
Val loss:   1.5185 | Val acc:   0.5763
Zapisano najlepszy model
Stage 2

Epoka 6/30
Train loss: 0.5489 | Train acc: 0.7929
Val loss:   1.5541 | Val acc:   0.5904
Zapisano najlepszy model

Epoka 7/30
Train loss: 0.4727 | Train acc: 0.8190
Val loss:   1.4323 | Val acc:   0.6100
Zapisano najlepszy model

Epoka 8/30
Train loss: 0.4130 | Train acc: 0.8373
Val loss:   1.4433 | Val acc:   0.6416
Zapisano najlepszy model

Epoka 9/30
Train loss: 0.3795 | Train acc: 0.8494
Val loss:   1.3017 | 

In [12]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

model.load_state_dict(
    torch.load(
        "/content/drive/MyDrive/runs_vit_b_16/model_best_vit_b_16.pth"
    )
)

model.eval()

true_labels = []
pred_labels = []

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        true_labels.extend(labels.cpu().numpy())
        pred_labels.extend(predicted.cpu().numpy())

In [13]:
runs_dir = "/content/drive/MyDrive/runs_vit_b_16"
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

accuracy = accuracy_score(true_labels, pred_labels)

precision = precision_score(
    true_labels,
    pred_labels,
    average='macro'
)

recall = recall_score(
    true_labels,
    pred_labels,
    average='macro'
)

f1 = f1_score(
    true_labels,
    pred_labels,
    average='macro'
)

print("\nWyniki modelu")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")

metrics_file = os.path.join(
    runs_dir,
    "test_metrics.txt"
)

with open(metrics_file, "w") as f:

    f.write("Wyniki modelu\n")
    f.write(f"Accuracy : {accuracy:.4f}\n")
    f.write(f"Precision: {precision:.4f}\n")
    f.write(f"Recall   : {recall:.4f}\n")
    f.write(f"F1-score : {f1:.4f}\n")

print(f"\nZapisano do: {metrics_file}")


Wyniki modelu
Accuracy : 0.8170
Precision: 0.8301
Recall   : 0.8151
F1-score : 0.8146

Zapisano do: /content/drive/MyDrive/runs_vit_b_16/test_metrics.txt


In [14]:
import seaborn as sns

from sklearn.metrics import confusion_matrix
runs_dir = "/content/drive/MyDrive/runs_vit_b_16"

os.makedirs(runs_dir, exist_ok=True)

# loss plot

plt.figure(figsize=(10,5))

plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Val Loss')

plt.title("Loss")
plt.xlabel('Epoch')
plt.ylabel('Loss')

plt.legend()

plt.savefig(
    os.path.join(runs_dir, "loss_plot.png"),
    bbox_inches='tight'
)

plt.close()

# accuracy plot

plt.figure(figsize=(10,5))

plt.plot(train_accuracies, label='Train Accuracy')
plt.plot(val_accuracies, label='Val Accuracy')

plt.title("Accuracy")
plt.xlabel('Epoch')
plt.ylabel('Accuracy')

plt.legend()

plt.savefig(
    os.path.join(runs_dir, "accuracy_plot.png"),
    bbox_inches='tight'
)

plt.close()

import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(
    true_labels,
    pred_labels
)

plt.figure(figsize=(14,12))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=test_dataset.classes,
    yticklabels=test_dataset.classes
)

plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")

plt.savefig(
    os.path.join(
        runs_dir,
        "test_confusion_matrix.png"
    ),
    bbox_inches="tight"
)

plt.close()

print("\nZapisano:")
print("- model_best.pth")
print("- model_current.pth")
print("- loss_plot.png")
print("- accuracy_plot.png")
print("- confusion_matrix.png")


Zapisano:
- model_best.pth
- model_current.pth
- loss_plot.png
- accuracy_plot.png
- confusion_matrix.png


In [ ]:
os.kill(os.getpid(), 9)